# Owned DQN and distributional targets

The owned networks use physical state features, legal masks, frozen targets and validation-only selection. Double selection, dueling values, prioritized replay and quantile regression are distinct extensions. Quantile crossing and failed seeds are retained.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/decision_lab").is_dir()
sys.path.insert(0, str(ROOT / "src"))


def read(path):
    return json.loads((ROOT / path).read_text())

## Double DQN target separation

In [2]:
import torch

from decision_lab.neural import quantile_huber, targets

r = torch.tensor([1.0])
term = torch.tensor([0.0])
online = torch.tensor([[2.0, 1.0]])
target = torch.tensor([[1.0, 9.0]])
legal = torch.tensor([[True, True]])
ordinary = targets(r, term, online, target, legal, 0.9, False)
double = targets(r, term, online, target, legal, 0.9, True)
assert torch.allclose(ordinary, torch.tensor([9.1])) and torch.allclose(double, torch.tensor([1.9]))
print("ordinary", ordinary, "double", double)

ordinary tensor([9.1000]) double tensor([1.9000])


## A quantile-loss hand oracle

In [3]:
loss = quantile_huber(torch.tensor([[0.0, 2.0]]), torch.tensor([[1.0, 3.0]]))
assert torch.allclose(loss, torch.tensor([0.3125]))
print(loss)

tensor([0.3125])


## Complete neural evidence

In [4]:
fits = pd.read_csv(ROOT / "docs/individual_fits.csv")
fits[fits.task == "Wireless online"][
    ["method", "seed", "test_return", "exact_normalized_gap", "shift_return"]
]

,method,seed,test_return,exact_normalized_gap,shift_return
48,double_dqn,0,-626.43,2.665448e-03,-819.77
49,double_dqn,1,-625.31,9.819304e-11,-819.68
50,double_dqn,2,-625.30,9.819358e-11,-819.77
51,double_dueling_per,0,-625.35,9.819319e-11,-819.73
52,double_dueling_per,1,-625.30,9.819358e-11,-819.77
53,double_dueling_per,2,-625.67,7.912140e-04,-819.72
54,dqn,0,-625.34,9.819315e-11,-819.73
55,dqn,1,-625.30,9.819401e-11,-819.74
56,dqn,2,-625.30,9.819358e-11,-819.77
57,qr_dqn,0,-625.31,9.819304e-11,-819.68


Full fits: `decision-lab neural --seed 0` (repeat seeds 1 and 2 where applicable). See [protocol](../docs/protocol.md), [results](../docs/results.md) and [references](../docs/references.md). These cells are numerical checks and evidence readers, not replacement training budgets.